In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from dbfread import DBF

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "Data"
    / "raw"
    / "Himalayan Database"
    / "HIMDATA"
)

# Load source data
peaks = pd.DataFrame(
    iter(DBF(DATA_PATH / "peaks.DBF", load=True))
)

exped = pd.DataFrame(
    iter(DBF(DATA_PATH / "exped.DBF", load=True))
)

print("Peaks shape      :", peaks.shape)
print("Expeditions shape:", exped.shape)

Peaks shape      : (503, 25)
Expeditions shape: (11854, 66)


In [2]:
peak_info = peaks[
    [
        "PEAKID",
        "PKNAME",
        "HEIGHTM",
        "LOCATION",
        "REGION",
        "OPEN",
        "TREKKING"
    ]
].copy()

model_data = exped.merge(
    peak_info,
    on="PEAKID",
    how="left"
)

print("Model dataset shape:", model_data.shape)

Model dataset shape: (11854, 72)


In [3]:
success_cols = [
    "SUCCESS1",
    "SUCCESS2",
    "SUCCESS3",
    "SUCCESS4"
]

model_data["EXPEDITION_SUCCESS"] = (
    model_data[success_cols].any(axis=1)
)

print(model_data["EXPEDITION_SUCCESS"].value_counts())

EXPEDITION_SUCCESS
True     6684
False    5170
Name: count, dtype: int64


In [4]:
pre_expedition_features = [
    "YEAR",
    "SEASON",
    "HEIGHTM",
    "REGION",
    "OPEN",
    "TREKKING",
    "TOTMEMBERS",
    "CAMPS"
]

X = model_data[pre_expedition_features].copy()
y = model_data["EXPEDITION_SUCCESS"].copy()

# Convert YEAR to numeric
X["YEAR"] = pd.to_numeric(
    X["YEAR"],
    errors="coerce"
)

# Convert boolean columns to integers
boolean_cols = [
    "OPEN",
    "TREKKING"
]

X[boolean_cols] = X[boolean_cols].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nMissing values:")
print(X.isnull().sum())

X shape: (11854, 8)
y shape: (11854,)

Features:
['YEAR', 'SEASON', 'HEIGHTM', 'REGION', 'OPEN', 'TREKKING', 'TOTMEMBERS', 'CAMPS']

Missing values:
YEAR          0
SEASON        0
HEIGHTM       0
REGION        0
OPEN          0
TREKKING      0
TOTMEMBERS    0
CAMPS         0
dtype: int64


In [5]:
from sklearn.model_selection import GroupShuffleSplit

groups = model_data["PEAKID"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print(
    "Training mountains:",
    model_data.iloc[train_idx]["PEAKID"].nunique()
)

print(
    "Testing mountains:",
    model_data.iloc[test_idx]["PEAKID"].nunique()
)

Training: (10740, 8)
Testing : (1114, 8)
Training mountains: 351
Testing mountains: 88


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

logistic_model = LogisticRegression(
    max_iter=3000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, logistic_pred))
print("Precision:", precision_score(y_test, logistic_pred))
print("Recall   :", recall_score(y_test, logistic_pred))
print("F1 Score :", f1_score(y_test, logistic_pred))

print("\nClassification Report:")
print(classification_report(y_test, logistic_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logistic_pred))

Accuracy : 0.6023339317773788
Precision: 0.5417867435158501
Recall   : 0.7504990019960079
F1 Score : 0.6292887029288703

Classification Report:
              precision    recall  f1-score   support

       False       0.70      0.48      0.57       613
        True       0.54      0.75      0.63       501

    accuracy                           0.60      1114
   macro avg       0.62      0.62      0.60      1114
weighted avg       0.63      0.60      0.60      1114


Confusion Matrix:
[[295 318]
 [125 376]]


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Accuracy : 0.644524236983842
Precision: 0.6003824091778203
Recall   : 0.626746506986028
F1 Score : 0.61328125

Classification Report:
              precision    recall  f1-score   support

       False       0.68      0.66      0.67       613
        True       0.60      0.63      0.61       501

    accuracy                           0.64      1114
   macro avg       0.64      0.64      0.64      1114
weighted avg       0.65      0.64      0.65      1114


Confusion Matrix:
[[404 209]
 [187 314]]


In [8]:
from sklearn.model_selection import GroupKFold, cross_validate

group_kfold = GroupKFold(n_splits=5)

rf_cv = cross_validate(
    rf_model,
    X,
    y,
    cv=group_kfold,
    groups=model_data["PEAKID"],
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1"
    ],
    n_jobs=-1
)

print("5-Fold Group Cross-Validation")
print("--------------------------------")

print(
    "Accuracy :",
    rf_cv["test_accuracy"].mean()
)

print(
    "Precision:",
    rf_cv["test_precision"].mean()
)

print(
    "Recall   :",
    rf_cv["test_recall"].mean()
)

print(
    "F1 Score :",
    rf_cv["test_f1"].mean()
)

5-Fold Group Cross-Validation
--------------------------------
Accuracy : 0.6332388385367133
Precision: 0.7019136374547449
Recall   : 0.5850199287884408
F1 Score : 0.6282137040505048


In [9]:
logistic_cv = cross_validate(
    logistic_model,
    X,
    y,
    cv=group_kfold,
    groups=model_data["PEAKID"],
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1"
    ],
    n_jobs=-1
)

print("5-Fold Group Cross-Validation - Logistic Regression")
print("----------------------------------------------------")

print(
    "Accuracy :",
    logistic_cv["test_accuracy"].mean()
)

print(
    "Precision:",
    logistic_cv["test_precision"].mean()
)

print(
    "Recall   :",
    logistic_cv["test_recall"].mean()
)

print(
    "F1 Score :",
    logistic_cv["test_f1"].mean()
)

5-Fold Group Cross-Validation - Logistic Regression
----------------------------------------------------
Accuracy : 0.674510552720208
Precision: 0.6942185925890978
Recall   : 0.7502959789856987
F1 Score : 0.7184047544504436


In [10]:
logistic_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.coef_[0]
}).sort_values(
    by="Coefficient",
    ascending=False
)

logistic_coefficients

,Feature,Coefficient
7,CAMPS,0.540892
5,TREKKING,0.303258
6,TOTMEMBERS,0.056401
0,YEAR,0.030133
2,HEIGHTM,-0.000524
1,SEASON,-0.044286
3,REGION,-0.221390
4,OPEN,-0.881584


In [11]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "CV Accuracy": [
        logistic_cv["test_accuracy"].mean(),
        rf_cv["test_accuracy"].mean()
    ],
    "CV Precision": [
        logistic_cv["test_precision"].mean(),
        rf_cv["test_precision"].mean()
    ],
    "CV Recall": [
        logistic_cv["test_recall"].mean(),
        rf_cv["test_recall"].mean()
    ],
    "CV F1": [
        logistic_cv["test_f1"].mean(),
        rf_cv["test_f1"].mean()
    ]
})

model_comparison.round(4)

,Model,CV Accuracy,CV Precision,CV Recall,CV F1
0,Logistic Regression,0.6745,0.6942,0.7503,0.7184
1,Random Forest,0.6332,0.7019,0.5850,0.6282


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = [
    "YEAR",
    "HEIGHTM",
    "TOTMEMBERS",
    "CAMPS"
]

categorical_features = [
    "SEASON",
    "REGION",
    "OPEN",
    "TREKKING"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=3000,
            random_state=42
        ))
    ]
)

In [13]:
from sklearn.model_selection import GroupKFold, cross_validate

group_kfold = GroupKFold(n_splits=5)

logistic_pipeline_cv = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=group_kfold,
    groups=model_data["PEAKID"],
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1"
    ],
    n_jobs=-1
)

print("Accuracy :", logistic_pipeline_cv["test_accuracy"].mean())
print("Precision:", logistic_pipeline_cv["test_precision"].mean())
print("Recall   :", logistic_pipeline_cv["test_recall"].mean())
print("F1 Score :", logistic_pipeline_cv["test_f1"].mean())

Accuracy : 0.6763378358466383
Precision: 0.6949282027534983
Recall   : 0.7364971163237339
F1 Score : 0.7136138070162639


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

logistic_pipeline.fit(X_train, y_train)

pipeline_pred = logistic_pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pipeline_pred))
print("Precision:", precision_score(y_test, pipeline_pred))
print("Recall   :", recall_score(y_test, pipeline_pred))
print("F1 Score :", f1_score(y_test, pipeline_pred))

print("\nClassification Report:")
print(classification_report(y_test, pipeline_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pipeline_pred))

Accuracy : 0.6113105924596051
Precision: 0.5550161812297735
Recall   : 0.6846307385229541
F1 Score : 0.613047363717605

Classification Report:
              precision    recall  f1-score   support

       False       0.68      0.55      0.61       613
        True       0.56      0.68      0.61       501

    accuracy                           0.61      1114
   macro avg       0.62      0.62      0.61      1114
weighted avg       0.62      0.61      0.61      1114


Confusion Matrix:
[[338 275]
 [158 343]]


In [15]:
final_comparison = pd.DataFrame({
    "Model": [
        "Majority Baseline",
        "Logistic Regression",
        "Random Forest",
        "Logistic Regression (Preprocessed)"
    ],
    "CV Accuracy": [
        0.5503,
        logistic_cv["test_accuracy"].mean(),
        rf_cv["test_accuracy"].mean(),
        logistic_pipeline_cv["test_accuracy"].mean()
    ],
    "CV Precision": [
        None,
        logistic_cv["test_precision"].mean(),
        rf_cv["test_precision"].mean(),
        logistic_pipeline_cv["test_precision"].mean()
    ],
    "CV Recall": [
        None,
        logistic_cv["test_recall"].mean(),
        rf_cv["test_recall"].mean(),
        logistic_pipeline_cv["test_recall"].mean()
    ],
    "CV F1": [
        None,
        logistic_cv["test_f1"].mean(),
        rf_cv["test_f1"].mean(),
        logistic_pipeline_cv["test_f1"].mean()
    ]
})

final_comparison.round(4)

,Model,CV Accuracy,CV Precision,CV Recall,CV F1
0,Majority Baseline,0.5503,NaN,NaN,NaN
1,Logistic Regression,0.6745,0.6942,0.7503,0.7184
2,Random Forest,0.6332,0.7019,0.5850,0.6282
3,Logistic Regression (Preprocessed),0.6763,0.6949,0.7365,0.7136


In [16]:
# Train final model on the complete dataset

final_model = logistic_pipeline.fit(X, y)

print("Final model trained successfully.")
print("Training samples:", len(X))
print("Features:", X.shape[1])

Final model trained successfully.
Training samples: 11854
Features: 8


In [17]:
import joblib

model_path = PROJECT_ROOT / "Data" / "processed" / "peakrisk_model.joblib"

joblib.dump(final_model, model_path)

print("Model saved successfully!")
print(model_path)

Model saved successfully!
c:\Projects\PeakRisk\Data\processed\peakrisk_model.joblib


In [18]:
def predict_expedition_success(
    year,
    season,
    heightm,
    region,
    open_status,
    trekking,
    total_members,
    camps
):
    input_data = pd.DataFrame([{
        "YEAR": year,
        "SEASON": season,
        "HEIGHTM": heightm,
        "REGION": region,
        "OPEN": open_status,
        "TREKKING": trekking,
        "TOTMEMBERS": total_members,
        "CAMPS": camps
    }])

    prediction = final_model.predict(input_data)[0]
    probability = final_model.predict_proba(input_data)[0]

    return {
        "prediction": bool(prediction),
        "success_probability": float(probability[1]),
        "failure_probability": float(probability[0])
    }

In [19]:
result = predict_expedition_success(
    year=2025,
    season=1,
    heightm=6500,
    region=1,
    open_status=1,
    trekking=0,
    total_members=10,
    camps=3
)

result

{'prediction': True,
 'success_probability': 0.8708009365900821,
 'failure_probability': 0.12919906340991794}